# R-Package Compatibility: `twfeweights`, `ptetools`, and `badcontrols`

This notebook walks end-to-end through the **three R-package translations** bundled
inside `diff_diff`. They import directly from `diff_diff`, and their APIs mirror the
original R packages closely, so you can move between R and Python without re-learning
the interface.

| R package      | diff-diff module        | Primary entry points                              |
|----------------|-------------------------|---------------------------------------------------|
| `twfeweights`  | `diff_diff.twfeweights` | `twfe_weights`, `ggtwfeweights`                    |
| `ptetools`     | `diff_diff.ptetools`    | `pte`, `two_by_two_subset`, `did_attgt`, `ggpte`   |
| `badcontrols`  | `diff_diff.badcontrols` | `didbc`, `simulate_bad_controls`, `extract_att`    |

We use real data already in the repo (`benchmarks/data/real/mpdta.csv`, the classic
Callaway--Sant'Anna county panel) for the first two packages and a small simulation
for the third.

---

## Setup

```bash
pip install -e ".[dev]"
```

> `est_method="imputation"` (Section 3) is dependency-light. The double-robust ML
> paths (`est_method="dr_ml"`) additionally need the optional extra
> `pip install -e ".[ml]"`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from diff_diff import (
    # ptetools
    pte, two_by_two_subset, did_attgt, ggpte,
    # twfeweights
    twfe_weights, ggtwfeweights,
    # badcontrols
    didbc, simulate_bad_controls, extract_att,
)

### Data: the Callaway--Sant'Anna county panel

`benchmarks/data/real/mpdta.csv` is a county x year panel. Column decoding:
`lemp` = log employment (outcome), `first.treat` = first year a staggered
minimum-wage policy took effect (`0` = never treated), `countyreal` = county id.
This is exactly the shape `pte` (group-time ATT) and `twfe_weights` (TWFE
decomposition) expect.


In [ ]:
from pathlib import Path

# Locate benchmarks/data/real/mpdta.csv no matter where the kernel was started:
# try the current dir, then walk a few levels up toward the repo root.
path = None
for depth in range(6):
    cand = Path.cwd().joinpath(*[".."] * depth, "benchmarks", "data", "real", "mpdta.csv")
    if cand.exists():
        path = cand
        break
if path is None:
    raise FileNotFoundError("run from the repo root -- benchmarks/data/real/mpdta.csv not found")

mpd = pd.read_csv(path)
print(f"loaded {len(mpd)} rows from {path}")
print(mpd.groupby("first.treat").size().rename("rows").to_string())

---

## 1. `ptetools` -- group-time ATT with `pte`

`pte` runs the generic group x time loop (R `compute.pte`) over a long staggered
panel. Point estimates are fast; set `bstrap=True` if you want bootstrap standard
errors (omitted here for speed).

In [ ]:
pet = pte(
    mpd,
    yname="lemp", gname="first.treat", tname="year", idname="countyreal",
)
print("overall ATT = %.4f (log employment)" % pet.overall_att)
print("\nATT(g, t) cells:")
pet.att_gt

`att_gt` holds the group x time ATT (columns `group` / `time` / `attgt` / `se`).
The overall ATT (~ -0.024 on log employment) matches the well-known estimate for this
dataset.

### A single `(g, t)` cell, directly

`two_by_two_subset()` isolates the balanced 2x2 comparison for one cohort and one
outcome year; `did_attgt()` runs the ATT estimator on that design. We use cohort 2004
at outcome year 2005.

In [ ]:
cell = two_by_two_subset(
    mpd, g=2004, tp=2005,
    gname="first.treat", tname="year", idname="countyreal", yname="lemp",
)
cell.gt_data.data[["id", "name", "Y", "D"]].head()

In [ ]:
att = did_attgt(cell.gt_data)
print("ATT(2004, 2005) = %.6f" % att.attgt)

(Event-study plot of the full PTE results; `ggpte` returns a matplotlib `Axes`.)

In [ ]:
ggpte(pet, show=False)
plt.title("Event study (ptetools::pte) -- county employment")
plt.show()

---

## 2. `twfeweights` -- decompose the TWFE estimator

`twfe_weights()` computes, for every ATT(g,t) cell, the weight the standard
two-way-fixed-effects (TWFE) regression implicitly places on it. A **negative weight
means TWFE can report the wrong sign** even when every treated (g,t) effect is
positive -- the de Chaisemartin--D'Haultfoeuille negative-weight problem.

It needs an ATT(g,t) table aligned to the panel's own column names (group column =
`first.treat`, calendar column = `year`) plus the panel.

In [ ]:
att_gt = pet.att_gt.rename(columns={"group": "first.treat", "time": "year"})
wts = twfe_weights(att_gt, mpd, group="first.treat", time="year",
                   treatment_group="first.treat")
wts.weights_df

In [ ]:
neg = int((wts.weights_df["weight"] < 0).sum())
print(f"{neg} of {len(wts.weights_df)} ATT(g,t) cells carry a NEGATIVE TWFE weight")
print("A plain TWFE event-study/recovery run on this panel can therefore report")
print("the wrong sign for those (g,t) cells.")

In [ ]:
ggtwfeweights(wts)
plt.show()

---

## 3. `badcontrols` -- "good" controls can be bad

`badcontrols` argues that conditioning on a pre-treatment covariate that is *itself
affected by treatment* (a bad control) can bias the estimate. `simulate_bad_controls()`
builds a two-period panel whose covariate `X` really is treatment-affected, so we have
a known `true_att` to check `didbc()` against.

In [ ]:
sim = simulate_bad_controls(n=800, seed=7)
panel = sim["data"]
print("simulated panel:", panel.shape, "columns:", list(panel.columns))
print("true overall ATT = %.4f" % sim["true_att_overall"])

In [ ]:
res = didbc(
    panel,
    yname="Y", gname="G", tname="period", idname="id",
    bad_control="X",
    est_method="imputation", seed=7,
)
extract_att(res)

The deterministic imputation estimate lands close to the simulated truth
(`true_att_overall` printed above). `res.method` records which estimator ran
(`imputation-staggered`). For a double-robust (cross-fitted) variant, install the ML
extra and use `est_method="dr_ml", nuisance_method="parametric"` with
`d_covariates=["Z"]`:

```python
res_ml = didbc(panel,
               yname="Y", gname="G", tname="period", idname="id",
               bad_control="X", d_covariates=["Z"],
               est_method="dr_ml", nuisance_method="parametric", seed=7)
extract_att(res_ml)
```


---

## Summary

| Package          | You called                                          | What you got                                                         |
|------------------|-----------------------------------------------------|----------------------------------------------------------------------|
| `ptetools`       | `pte()` (or `two_by_two_subset` + `did_attgt` + `ggpte`) | staggered group-time ATT + event study on a real panel          |
| `twfeweights`    | `twfe_weights()` + `ggtwfeweights()`                | TWFE cell weights / negative-weight diagnostic                        |
| `badcontrols`    | `didbc()` on `simulate_bad_controls()`              | bad-control-aware ATT vs a known truth                                |

All three are drop-in translations of the R packages, so a pipeline you already run in
R (`pte` + `twfe_weights`, or `didbc`) can be reproduced in Python with the same call
shape. Parity notes and known deviations live in `docs/methodology/REGISTRY.md`.